# NFL_PLAYER_PASSING_TDS

Passing TD count. Not a Bernoulli anytime-TD model.

This notebook is the full v1 loop: look at the data, **train** a model, **save** it in Snowflake, **predict** every row, then **measure** how wrong we were on 2025. Run cells in order. If you only want the whole loop in one shot, skip to `cook` at the bottom — but the first time, walk through.

**What we predict:** a **number**: `label_passing_tds`.
**Grain:** one row per player-game. Inputs come from rolling rates + weather products; the label is joined from the CORE box score so we never train on the current game's yards.
**Split:** train on 2023–24 regular season + playoffs; test on 2025. That is walk-forward (time order), not a random shuffle.

**Words you will see:**

- **X / features** — columns the model is allowed to use. Never `close_*` (betting lines) and never `label_*` (the answer).
- **y / label** — the answer for completed games. Null on unplayed games.
- **Fit** — learn patterns from 2023–24.
- **Score** — write a prediction for every row, including next week's slate.
- **Register** — store the fitted model in `NFL_PROD_DB.ML` so Snowflake can run it later.

Keep the repo `ml/` folder in this Workspace. Run the install cell after a service restart. Do not promote a version just because it exists. Do not add FEATURES or ML tables to the Cortex agents.

## Install the ML libraries

A model is just math that finds patterns in a table. We use two libraries:

- **scikit-learn** — trains the model on your laptop-shaped Python process (the notebook kernel).
- **snowflake-ml-python** — saves that trained model into Snowflake's Model Registry and can score it on a compute pool later.

This cell installs them with `uv pip` against Snowflake's own PyPI mirror (not public pypi.org, not Anaconda). The base image often already has them, but a weekend service restart wipes extra installs, so run this first every session. The `print` lines confirm versions so a later error is not a mystery missing-package.

In [ ]:
!uv pip install scikit-learn snowflake-ml-python

import importlib.metadata as md

print("sklearn", md.version("scikit-learn"))
print("snowflake-ml-python", md.version("snowflake-ml-python"))

## Count the player-games we can actually learn from

Player models sit on **one row per player per game**. Features come from `FEAT_PLAYER_GAME_ROLLING` (trailing rates) plus a few weather products. The **answer** (`label_passing_tds`) does **not** live on that rolling table — current-game yards would leak the thing we are trying to predict. Labels are joined later from `CORE.FACT_PLAYER_GAME_OFFENSE`.

The **gate** `n_passing_l5 > 0` means: this player actually did that thing in the trailing window. A receiver with zero recent targets is not a useful receiving-yards example. Week 1 of a season often fails the gate (no trailing games yet). That is correct, not a bug.

- **`rows`** — every player-game stub, including unplayed slates.
- **`completed_rs_post`** — finished regular-season / postseason rows.
- **`gated`** — those rows that also pass the role gate. Training uses gated.

In [ ]:
%%sql -r player_grain
SELECT
    COUNT(*) AS rows,
    COUNT(IFF(r.is_completed AND r.season_type IN (2, 3), 1, NULL)) AS completed_rs_post,
    COUNT(IFF(r.is_completed AND r.season_type IN (2, 3) AND r.n_passing_l5 > 0, 1, NULL)) AS gated
FROM NFL_PROD_DB.FEATURES.FEAT_PLAYER_GAME_ROLLING r

## Confirm each season has gated player-games

Same walk-forward idea as the game models: **train 2023–24, test 2025**. This count is players who passed `n_passing_l5 > 0`, not every roster name. A collapsed 2025 bar means the holdout is empty and `fit` will error.

In [ ]:
%%sql -r eligible_by_season
SELECT r.season, COUNT(*) AS n
FROM NFL_PROD_DB.FEATURES.FEAT_PLAYER_GAME_ROLLING r
WHERE r.is_completed AND r.season_type IN (2, 3)
  AND r.n_passing_l5 > 0
GROUP BY 1
ORDER BY 1

## Connect the notebook to Snowflake and load this model's recipe

Two things happen here:

1. **Find `weekend_warriors_ml`.** The training code lives in the repo's `ml/` folder, not inside this notebook. We walk up from the current directory until we see `weekend_warriors_ml/pipeline.py`, then put that folder on `sys.path` so `import` works. If this raises, the Workspace is missing the `ml/` folder from the git pull.
2. **Open the kernel session.** `get_active_session()` is the Snowflake login this notebook already has. We do not type a password here.

`get_spec("NFL_PLAYER_PASSING_TDS")` loads the recipe: which table, which columns are X (inputs), which column is the label (the answer we want to predict), and whether this is regression (a number) or classification (a yes/no probability). Nothing is trained yet.

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for cand in (here, *here.parents):
    if (cand / "weekend_warriors_ml" / "pipeline.py").exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
else:
    raise FileNotFoundError(
        "weekend_warriors_ml not found. Put the repo ml/ folder in this Workspace."
    )

from snowflake.snowpark.context import get_active_session
from weekend_warriors_ml.specs import get_spec
from weekend_warriors_ml.pipeline import (
    cook,
    fit,
    inspect,
    log_experiment,
    register,
    score_batch,
    score_local,
)

SPEC = get_spec("NFL_PLAYER_PASSING_TDS")
session = get_active_session()
print(SPEC.name, SPEC.task, len(SPEC.feature_columns), "features")
print(session.get_current_role(), session.get_current_warehouse())

## Same counts, through the Python pipeline

`inspect` runs the **same filters the trainer will use**, then prints a JSON summary. Read it as a sanity check before you spend time fitting:

- **`rows`** — pulled into pandas.
- **`eligible_rs_post`** — completed RS+post with a label and `n_passing_l5 > 0`.
- **`by_season`** — must include 2023, 2024, and 2025.
- **`n_features`** — width of X. None of those names start with `close_` or `label_`.

If this disagrees with the SQL cells above, stop and look at the join — do not fit a broken frame.

In [ ]:
inspect(session, SPEC.name)

## Train the model, then grade it on 2025

This is **regression**: we predict a number (`label_passing_tds`). **X** is the feature columns (rest, weather, trailing rates). **y** is the label. The model never sees `close_*` or `label_*` in X.

**HistGradientBoostingRegressor** is a tree ensemble: many small decision trees, each correcting the last. Depth 6 and 200 iterations are fixed for v1 so every notebook is comparable. We are not searching hyperparameters here.

What the printed metrics mean (all on **2025 only**, the holdout):

- **`mae`** — mean absolute error. "On average we were off by this many units" (points, yards, …).
- **`rmse`** — same idea, but big misses hurt more.
- **`mae_mean_baseline`** — error if we predicted the **training-set average** every time. If `mae` is not better than this, the model is not yet beating "guess the mean." That is information, not a failure of the notebook.

The function returns `(model, metrics)`. `model` is the fitted object we register and score next.

In [ ]:
model, metrics = fit(session, SPEC.name)
metrics

## Save the run and register the model in Snowflake

Two writes, one cell:

1. **`log_experiment`** — stores the hyperparameters and the 2025 metrics on an experiment named after this model. If tracking is unavailable, we print the error and continue. Metrics still exist in the cell output.
2. **`register` / `log_model`** — uploads the fitted sklearn object to `NFL_PROD_DB.ML.NFL_PLAYER_PASSING_TDS`. Snowflake mints a `version_name` (often a random animal). We pin `pip_requirements=["scikit-learn"]` and `target_platforms` to Snowpark Container Services so this cannot silently become a warehouse Python UDF.

Copy `version_name`. The next cells need it. Registering is **not** promoting — nobody should treat this as production because the object exists.

In [ ]:
try:
    log_experiment(session, metrics, SPEC.name)
except Exception as exc:
    print("Experiment tracking skipped:", exc)

version_name = register(session, model, SPEC.name)
version_name

## Confirm the model object exists

`SHOW MODELS` lists registry entries. You should see `NFL_PLAYER_PASSING_TDS`. Aliases like default / first / last are Snowflake bookkeeping. This is only a listing — it does not mean the model is good.

In [ ]:
%%sql -r models
SHOW MODELS LIKE 'NFL_PLAYER_PASSING_TDS' IN SCHEMA NFL_PROD_DB.ML

## Confirm Snowflake exposed a predict function

A registered model publishes functions (usually `PREDICT`). The `{{version_name}}` token is this notebook's interpolation of the Python variable from the cell above — run that cell first. Empty output means the version string did not bind.

In [ ]:
%%sql -r model_fns
SHOW FUNCTIONS IN MODEL NFL_PROD_DB.ML.NFL_PLAYER_PASSING_TDS VERSION {{version_name}}

## Predict every row and write the pred table

Scoring is "run the fitted model on X for the **full slate**." Completed games get a prediction **and** a label so we can measure error. Unplayed games get a prediction and a null label — that is next week's number.

Writes `NFL_PROD_DB.ML.PRED_PLAYER_PASSING_TDS` (create-or-replace). The prediction column is `pred_passing_tds` (same units as the label). `preds.head(5)` is a peek, not the evaluation.

In [ ]:
preds = score_local(session, model, version_name, SPEC.name)
preds.head(5)

## Did the pred table land?

`NFL_PROD_DB.ML.PRED_PLAYER_PASSING_TDS` should now exist (created on first write).

- **`rows`** — full slate we scored, including unplayed.
- **`labeled`** — rows where we know the answer (completed games).
- **`version_name`** — the registry version this write came from.

A zero-row table means `write_pandas` did not run or pointed at another schema.

In [ ]:
%%sql -r pred_summary
SELECT
    COUNT(*) AS rows,
    COUNT(label_passing_tds) AS labeled,
    MIN(scored_at) AS scored_at,
    MIN(version_name) AS version_name
FROM NFL_PROD_DB.ML.PRED_PLAYER_PASSING_TDS

## Grade 2025 from the pred table

This is the number you quote: **how far off were we on games the model had never seen?**

- **`mae`** — should be close to `fit`'s printed MAE.
- **`mse`** — mean squared error (RMSE is the square root of this).

We filter to 2025 regular season + postseason with a known label. Unplayed 2026 rows are excluded here on purpose — there is no truth to compare.

In [ ]:
%%sql -r walkforward
SELECT
    COUNT(*) AS n_2025_rs_post,
    ROUND(AVG(ABS(pred_passing_tds - label_passing_tds)), 3) AS mae,
    ROUND(AVG(POW(pred_passing_tds - label_passing_tds, 2)), 3) AS mse
FROM NFL_PROD_DB.ML.PRED_PLAYER_PASSING_TDS
WHERE season = 2025
  AND season_type IN (2, 3)
  AND label_passing_tds IS NOT NULL

## Optional: score again on the ML compute pool

Everything above used the **notebook kernel**. `score_batch` asks Snowflake to run the **registered** version on `ML_DEV_POOL` (`CPU_X64_S`, not `DLT_POOL`) and drop files on the inference stage. Use this to prove the registry object works in a container. Skip it if you only needed the pred table — it takes several minutes and starts a pool node.

This is not required to finish v1.

In [ ]:
score_batch(session, version_name, SPEC.name)

## One-cell path (same pipeline, no pool)

`cook` is inspect → fit → log experiment → register → `score_local`. It does **not** call `score_batch`. Use this when an agent is driving the notebook and you already understand the steps above. Running both the step-by-step cells **and** `cook` will train twice and overwrite the pred table — pick one path per session.

In [ ]:
cook(session, SPEC.name)